In [1]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "1ETOJv6UCQP8Azj_v83KZKjwPZibXkTGgC0UvTTUG5hY"
SHEET_NAME = "&mica"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()


# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [2]:
# List all sheets (tabs) in the spreadsheet via the Google Sheets API.
# DuckDB's gsheets extension can only read one named tab at a time, so we
# use gspread (same service-account creds) to discover the tab names.
gc = gspread.service_account(filename=str(credentials_path.resolve()))
spreadsheet = gc.open_by_key(SPREADSHEET_ID)

sheet_names = [ws.title for ws in spreadsheet.worksheets()]
sheet_names

['uberregional kunde',
 'template',
 '&mica',
 'allO Tech',
 'Balticsecur',
 'BDKplan',
 'Bernstein',
 'Beyond Imaging',
 'Bucherer',
 'C&C R Nord',
 'C&C Service',
 'Cornelsen',
 'EJF_IS',
 'eps',
 'EUREF',
 'FERI',
 'Fieldfisher',
 'GFSUDIG',
 'Gn Bauphysik',
 'Giga Green',
 'HeyJobs',
 'iwoca',
 'innoscripta',
 'Jebsen & Jessen',
 'Juit',
 'K&L Gates',
 'Kalialab',
 'Kienbaum',
 'KMG',
 'Medicover_IS',
 'LIQID Asset',
 'LIQID Investment',
 'MOIA',
 'Okaidi',
 'S-thetic',
 'Serviceware',
 'Syngenio',
 'Uni Elektro',
 'X1F',
 'Zentrum Beatmung']

In [ ]:
# Build a tidy table from every client tab (one row per entity line).
#
# Two normalizations happen here:
#   1. Header labels drift between tabs -> map every variant to a canonical name.
#   2. padoa_id is split into (mother_id, padoa_id) so the pair links each entity
#      to its accounting collection:
#        '130000478_new_3' -> mother_id=130000478, padoa_id=new_3   (mother explicit in string)
#        '113010031'       -> mother_id=113010031, padoa_id=113010031 (a formatted id IS its own mother)
#        'new_1'           -> mother_id=<tab mother>, padoa_id=new_1 (mother inferred per tab)
#      A tab's mother = the row whose padoa_id is pure-numeric AND acc_coll_mother
#      or admin_shell is TRUE, used only to resolve the 'new_n' rows. Tabs with 0 or
#      >1 candidate mothers can't resolve those, so they get mother_id=NULL and needs_review=TRUE.
#
# All client tabs are fetched in ONE batch request to stay under the read quota.
import re

# index + blank template, plus tabs we deliberately skip
META_TABS = {"uberregional kunde", "template"}
EXCLUDE_TABS = {"EJF_IS", "Medicover_IS"}
SKIP_TABS = META_TABS | EXCLUDE_TABS

HEADER_MAP = {
    "source": "source",
    "id": "source_id",
    "name": "name",
    "standort": "standort",
    "address": "address",
    "padoa_id": "padoa_raw",
    "acc.coll mother": "acc_coll_mother",
    "acc.coll daughter": "acc_coll_daughter",
    "simple coll": "simple_coll",
    "sim.coll": "simple_coll",
    "admin shell (basic care invoicing)": "admin_shell",
    "admin shell": "admin_shell",
    "merge into": "merge_into",
    "comment": "comment",
    # stray "Basic Care" column (1 tab) left unmapped -> dropped
}
BOOL_COLS = ["acc_coll_mother", "acc_coll_daughter", "simple_coll", "admin_shell"]
TEXT_COLS = ["source", "source_id", "name", "standort", "address", "comment"]

RE_MOTHER_NEW = re.compile(r"^(\d+)_new_\d+$", re.I)   # <mother>_new_n
RE_NEW = re.compile(r"^new_\d+$", re.I)                # new_n
RE_PURE = re.compile(r"^\d+$")                         # pure numeric id


def _to_bool(v):
    v = (v or "").strip().upper()
    return True if v == "TRUE" else False if v == "FALSE" else None


def _as_str(v):
    # blanks stored as None come back as NaN (float) after the DataFrame build
    return v if isinstance(v, str) else ""


# --- fetch every client tab in one batch request ---
client_titles = [t for t in sheet_names if t not in SKIP_TABS]
value_ranges = spreadsheet.values_batch_get(
    [f"'{t}'!A1:AB200" for t in client_titles]
)["valueRanges"]

records = []
for title, vr in zip(client_titles, value_ranges):
    rows = vr.get("values", [])
    if not rows:
        continue
    col_at = {}  # canonical name -> column index (first wins)
    for i, h in enumerate(rows[0]):
        canon = HEADER_MAP.get(h.strip().lower())
        if canon and canon not in col_at:
            col_at[canon] = i
    for r in rows[1:]:
        if not any(c.strip() for c in r):
            continue

        def cell(canon):
            i = col_at.get(canon)
            return r[i].strip() if i is not None and i < len(r) else ""

        rec = {"client_group": title, "padoa_raw": cell("padoa_raw") or None}
        for c in TEXT_COLS:
            rec[c] = cell(c) or None
        for c in BOOL_COLS:
            rec[c] = _to_bool(cell(c))
        records.append(rec)

df_entities = pd.DataFrame.from_records(records)

# --- drop template scaffolding rows (only boolean flags, no identifying data) ---
n_raw = len(df_entities)
has_ident = (
    df_entities["source_id"].notna()
    | df_entities["name"].notna()
    | df_entities["padoa_raw"].notna()
)
df_entities = df_entities[has_ident].copy()
print(f"Dropped {n_raw - len(df_entities)} scaffolding rows -> {len(df_entities)} entity rows")

# --- find each tab's mother (pure-numeric padoa_id flagged as mother/admin shell) ---
# Only used to resolve 'new_n' rows, which carry no id of their own.
def _is_mother_cand(row):
    p = _as_str(row["padoa_raw"])
    return bool(
        p and RE_PURE.match(p) and (bool(row["acc_coll_mother"]) or bool(row["admin_shell"]))
    )

df_entities["_mother_cand"] = df_entities.apply(_is_mother_cand, axis=1)
tab_mother = {}
for tab, g in df_entities.groupby("client_group"):
    cands = sorted(set(g.loc[g["_mother_cand"], "padoa_raw"]))
    tab_mother[tab] = cands[0] if len(cands) == 1 else None  # None if 0 or >1 (ambiguous)

# --- split padoa_raw -> (mother_id, padoa_id, needs_review) ---
def _split(row):
    p = _as_str(row["padoa_raw"])
    mother = tab_mother.get(row["client_group"])
    if not p:
        return pd.Series([None, None, False])
    m = RE_MOTHER_NEW.match(p)
    if m:                                          # mother explicit in string
        return pd.Series([m.group(1), p.split("_", 1)[1], False])
    if RE_PURE.match(p):                           # already a formatted id -> it IS the mother_id
        return pd.Series([p, p, False])
    if RE_NEW.match(p):                            # new_n daughter -> inferred tab mother
        return pd.Series([mother, p, mother is None])
    return pd.Series([None, p, True])              # n/a, "new", id-lists -> manual review

df_entities[["mother_id", "padoa_id", "needs_review"]] = df_entities.apply(_split, axis=1)

# --- final column order ---
df_entities = df_entities[
    ["client_group", "source", "source_id", "name", "standort", "address",
     "mother_id", "padoa_id", "needs_review"] + BOOL_COLS + ["comment"]
]

print(f"needs_review rows: {int(df_entities['needs_review'].sum())} "
      f"| mother_id resolved: {int(df_entities['mother_id'].notna().sum())}/{len(df_entities)}")
df_entities.head(10)

In [ ]:
# Write the tidy client data to Postgres (attached as `pg`) via DuckDB.
# Booleans are already real True/False/None, so DuckDB -> postgres keeps BOOLEAN types.
# (mother_id, padoa_id) is the couple that links each entity to its collection.
PG_TABLE = "pg.bas_firms.uberregional_entity"

# pandas 3 stores text as a 'str' dtype DuckDB's scanner rejects; coerce to
# plain object/None so DuckDB maps columns to VARCHAR/BOOLEAN cleanly.
_df_load = df_entities.astype(object).where(pd.notnull(df_entities), None)
duck.register("df_entities", _df_load)
duck.execute(f"""
    CREATE OR REPLACE TABLE {PG_TABLE} AS
    SELECT
        client_group,
        source,
        source_id,
        name,
        standort,
        address,
        mother_id,
        padoa_id,
        needs_review,
        acc_coll_mother,
        acc_coll_daughter,
        simple_coll,
        admin_shell,
        comment,
        now() AS loaded_at
    FROM df_entities;
""")

n = duck.sql(f"SELECT count(*) FROM {PG_TABLE}").fetchone()[0]
print(f"Wrote {n} rows to {PG_TABLE}")

# sanity: collection sizes via the (mother_id, padoa_id) couple + review backlog
duck.sql(f"""
    SELECT
        count(*)                                    AS rows,
        count(DISTINCT mother_id)                   AS collections,
        count(*) FILTER (WHERE mother_id IS NULL)   AS no_mother,
        count(*) FILTER (WHERE needs_review)        AS needs_review
    FROM {PG_TABLE}
""").df()

In [ ]:
# Append the "no manual check" clients from the master 'uberregional kunde' sheet as
# accounting collections, sourced from cross_regional_firms.
#   - one mother row per easybill ID (source='easybill', acc_coll_mother, padoa_id = easybill ID)
#   - each cross_regional_firms entity as a daughter (source='basic_care', acc_coll_daughter,
#     padoa_id = 'new_<n>' numbered per collection)
# Idempotent: these mother_ids never come from the client tabs, so we delete+reinsert them.
master = spreadsheet.worksheet("uberregional kunde").get_all_values()
mh = {h.strip(): i for i, h in enumerate(master[0])}
clean = [
    {
        "easybill_id": r[mh["easybill ID"]].strip(),
        "client_group": r[mh["client_group"]].strip(),
        "name": r[mh["Kunde"]].strip(),
    }
    for r in master[1:]
    if any(c.strip() for c in r) and r[mh["need_manual_check"]].strip().upper() == "FALSE"
]
df_clean = pd.DataFrame(clean)
df_clean = df_clean.astype(object).where(pd.notnull(df_clean), None)
duck.register("clean_master", df_clean)
print(f"clean (no-manual-check) clients: {len(df_clean)}")

duck.execute(f"DELETE FROM {PG_TABLE} WHERE mother_id IN (SELECT easybill_id FROM clean_master)")
duck.execute(f"""
INSERT INTO {PG_TABLE}
    (client_group, source, source_id, name, standort, address, mother_id, padoa_id,
     needs_review, acc_coll_mother, acc_coll_daughter, simple_coll, admin_shell, comment, loaded_at)
SELECT client_group, 'easybill', easybill_id, name,
       CAST(NULL AS VARCHAR), CAST(NULL AS VARCHAR),
       easybill_id, easybill_id,
       FALSE, TRUE, FALSE, FALSE, FALSE, CAST(NULL AS VARCHAR), now()
FROM clean_master
UNION ALL
SELECT m.client_group, 'basic_care', c."Wochenliste ID", c."Kunde",
       c."BAS Standort", c."Anschrift",
       m.easybill_id,
       'new_' || row_number() OVER (PARTITION BY c."easybill ID" ORDER BY c."Wochenliste ID"),
       FALSE, FALSE, TRUE, FALSE, FALSE, CAST(NULL AS VARCHAR), now()
FROM pg.bas_firms.cross_regional_firms c
JOIN clean_master m ON c."easybill ID" = m.easybill_id
""")

duck.sql(f"""
    SELECT
        count(*) AS total_rows,
        count(*) FILTER (WHERE mother_id IN (SELECT easybill_id FROM clean_master)
                         AND acc_coll_mother)   AS added_mothers,
        count(*) FILTER (WHERE mother_id IN (SELECT easybill_id FROM clean_master)
                         AND acc_coll_daughter) AS added_daughters
    FROM {PG_TABLE}
""").df()

In [ ]:
# Append the EJF_IS and Medicover_IS tabs (excluded from the main build above) WITHOUT
# rebuilding the table. Reuses the shared helpers/regexes from the build cell. Idempotent:
# scoped delete+insert on just these two client_groups, so existing rows are untouched.
LATE_TABS = ["EJF_IS", "Medicover_IS"]

_vr = spreadsheet.values_batch_get([f"'{t}'!A1:AB300" for t in LATE_TABS])["valueRanges"]
_records = []
for title, vr in zip(LATE_TABS, _vr):
    rows = vr.get("values", [])
    if not rows:
        continue
    col_at = {}
    for i, h in enumerate(rows[0]):
        canon = HEADER_MAP.get(h.strip().lower())
        if canon and canon not in col_at:
            col_at[canon] = i
    for r in rows[1:]:
        if not any(c.strip() for c in r):
            continue

        def cell(canon):
            i = col_at.get(canon)
            return r[i].strip() if i is not None and i < len(r) else ""

        rec = {"client_group": title, "padoa_raw": cell("padoa_raw") or None}
        for c in TEXT_COLS:
            rec[c] = cell(c) or None
        for c in BOOL_COLS:
            rec[c] = _to_bool(cell(c))
        _records.append(rec)

df_late = pd.DataFrame.from_records(_records)
df_late = df_late[
    df_late["source_id"].notna() | df_late["name"].notna() | df_late["padoa_raw"].notna()
].copy()

df_late["_mother_cand"] = df_late.apply(_is_mother_cand, axis=1)
_late_mother = {}
for tab, g in df_late.groupby("client_group"):
    cands = sorted(set(g.loc[g["_mother_cand"], "padoa_raw"]))
    _late_mother[tab] = cands[0] if len(cands) == 1 else None

def _split_late(row):
    p = _as_str(row["padoa_raw"])
    mother = _late_mother.get(row["client_group"])
    if not p:
        return pd.Series([None, None, False])
    m = RE_MOTHER_NEW.match(p)
    if m:
        return pd.Series([m.group(1), p.split("_", 1)[1], False])
    if RE_PURE.match(p):
        return pd.Series([p, p, False])
    if RE_NEW.match(p):
        return pd.Series([mother, p, mother is None])
    return pd.Series([None, p, True])

df_late[["mother_id", "padoa_id", "needs_review"]] = df_late.apply(_split_late, axis=1)
df_late = df_late[
    ["client_group", "source", "source_id", "name", "standort", "address",
     "mother_id", "padoa_id", "needs_review"] + BOOL_COLS + ["comment"]
]

duck.register("df_late", df_late.astype(object).where(pd.notnull(df_late), None))
duck.execute(f"DELETE FROM {PG_TABLE} WHERE client_group IN ('EJF_IS', 'Medicover_IS')")
duck.execute(f"""
    INSERT INTO {PG_TABLE}
        (client_group, source, source_id, name, standort, address, mother_id, padoa_id,
         needs_review, acc_coll_mother, acc_coll_daughter, simple_coll, admin_shell, comment, loaded_at)
    SELECT client_group, source, source_id, name, standort, address, mother_id, padoa_id,
           needs_review, acc_coll_mother, acc_coll_daughter, simple_coll, admin_shell, comment, now()
    FROM df_late;
""")
print(f"Appended {len(df_late)} rows for {LATE_TABS}")
duck.sql(f"""
    SELECT client_group, count(*) AS n,
           count(*) FILTER (WHERE needs_review) AS needs_review,
           count(*) FILTER (WHERE mother_id IS NULL) AS no_mother
    FROM {PG_TABLE} WHERE client_group IN ('EJF_IS', 'Medicover_IS') GROUP BY 1
""").df()